In [ ]:
import sys
sys.path.append('../scripts')
from build_train_manifest import classify_future_maneuver, MANEUVER_RANK

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import random
import numpy as np
from collections import defaultdict
from standard_e2e import Modality, TrajectoryComponent

TRAIN_DIR = '../data/processed/waymo_e2e/training/'

files = sorted([f for f in os.listdir(TRAIN_DIR) if f.endswith('.npz')])
print(f'Found {len(files):,} frame files')

Found 415,663 frame files


In [3]:
# group frames by sequence
sequences = defaultdict(list)
for fname in files:
    seq_id = fname.rsplit('_', 1)[0]
    sequences[seq_id].append(fname)

random.seed(42)
N_SEQUENCES = 200
sampled_seq_ids = random.sample(list(sequences.keys()), N_SEQUENCES)
print(f'Sampled {N_SEQUENCES} sequences (seed=42)')

Sampled 200 sequences (seed=42)


In [4]:
# for each sampled sequence: highest-ranked maneuver + Waymo intent
sequence_labels = {}  # seq_id -> (maneuver, intent)

for seq_id in sampled_seq_ids:
    best_maneuver = None
    best_rank = -1
    seq_intent = None

    for fname in sorted(sequences[seq_id]):
        data = np.load(os.path.join(TRAIN_DIR, fname), allow_pickle=True)
        modality = data['_modality_data'].item()

        # intent is constant within a sequence, read once
        if seq_intent is None:
            seq_intent = modality[Modality.INTENT].name

        future = modality[Modality.FUTURE_STATES]
        if future.isEmpty:
            continue
        xs = future.get(TrajectoryComponent.X).flatten()
        ys = future.get(TrajectoryComponent.Y).flatten()
        maneuver = classify_future_maneuver(xs, ys)
        rank = MANEUVER_RANK[maneuver]
        if rank > best_rank:
            best_rank = rank
            best_maneuver = maneuver

    if best_maneuver is not None:
        sequence_labels[seq_id] = (best_maneuver, seq_intent)

print(f'Labeled {len(sequence_labels)} sequences')

Labeled 200 sequences


## Cross-tabulation: Waymo intent vs. our maneuver labels

Each row is a Waymo `intent` value; each column is our derived maneuver. Cells show the
percentage of sequences with that intent that we assigned to each maneuver.

In [5]:
all_maneuvers = ['straight', 'stationary', 'left-turn', 'right-turn',
                 'lane-change-left', 'lane-change-right']

cross_tab = defaultdict(lambda: defaultdict(int))
for seq_id, (maneuver, intent) in sequence_labels.items():
    cross_tab[intent][maneuver] += 1

print(f"{'intent':<20}", "  ".join(f"{m[:14]:>14}" for m in all_maneuvers))
print("-" * 115)
for intent, counts in sorted(cross_tab.items()):
    total = sum(counts.values())
    row = "  ".join(f"{counts.get(m, 0)/total*100:>13.1f}%" for m in all_maneuvers)
    print(f"{intent:<20}  {row}  (n={total})")

intent                     straight      stationary       left-turn      right-turn  lane-change-le  lane-change-ri
-------------------------------------------------------------------------------------------------------------------
GO_LEFT                        22.7%            9.1%           63.6%            0.0%            0.0%            4.5%  (n=22)
GO_RIGHT                       25.0%            0.0%            8.3%           66.7%            0.0%            0.0%  (n=12)
GO_STRAIGHT                    52.4%            0.0%           16.9%           18.1%            7.8%            4.8%  (n=166)
